In [20]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

In [34]:
train = pd.read_csv('/content/train (1).csv')

/tmp/ipykernel_2008/2082494160.py:1: DtypeWarning: Columns (22) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv('/content/train (1).csv')


In [3]:
train.head()

,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,...,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class,satisfaction
0,0,36,694,4,4.0,4.0,3.0,1.0,4.0,1.0,...,4.0,3.0,1.0,0.0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
1,1,56,651,5,5.0,5.0,5.0,5.0,4.0,4.0,...,5.0,4.0,3.0,0.0,0.0,Male,Loyal Customer,Business travel,Business,True
2,2,31,1371,3,4.0,2.0,5.0,3.0,3.0,3.0,...,4.0,5.0,3.0,0.0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
3,3,10,1616,3,5.0,3.0,3.0,3.0,3.0,3.0,...,3.0,4.0,3.0,0.0,0.0,Male,Loyal Customer,Personal Travel,Eco,False
4,4,23,481,3,4.0,3.0,4.0,5.0,3.0,5.0,...,5.0,3.0,5.0,5.0,0.0,Female,disloyal Customer,Business travel,Eco,False


In [6]:
train.dtypes

,0
id,int64
Age,int64
Flight Distance,int64
Inflight wifi service,int64
Departure/Arrival time convenient,float64
Ease of Online booking,float64
Gate location,float64
Food and drink,float64
Online boarding,float64
Seat comfort,float64


In [7]:
X = train.drop(['id', 'satisfaction'], axis=1)
y = train['satisfaction']

In [8]:
num_features = X.select_dtypes(exclude='object').columns
cat_features = X.select_dtypes(include='object').columns

In [11]:
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder())
])

In [10]:
le = LabelEncoder()
y = le.fit_transform(y)

In [12]:
preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_features),
    ('cat', cat_pipeline, cat_features)
])

In [13]:
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier())
])

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [15]:
full_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['Age', 'Flight Distance', 'Inflight wifi service',
       'Departure/Arrival time convenient', 'Ease of Online booking',
       'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort',
       'Inflight entertainm...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=None,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=None, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=None, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [17]:
y_pred=full_pipeline.predict(X_test)

In [36]:
test = pd.read_csv('/content/test.csv')
X_kaggle = test.drop(columns=['id'])
y_pred = full_pipeline.predict(X_kaggle)
submission = pd.DataFrame({
    'id': test['id'],
    'target': y_pred
})
submission.to_csv('/content/submission.csv', index=False)

print(submission.shape)
print(submission.head())

(299844, 2)
       id  target
0  699635       0
1  699636       1
2  699637       1
3  699638       0
4  699639       0
